[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/059.%20The%20Warehouse%20Layout%20Design%20Problem/P59-Tier-2_executed.ipynb)

---


# 59. The Warehouse Layout Design Problem

## Tier 2: Randomized Construction with Pairwise-Swap Improvement

### Purpose and inputs

This notebook assigns functional areas to candidate warehouse locations. Its inputs are the areas, their space requirements, location capacities and coordinates, and directed material-flow volumes between areas. Its output is the best feasible assignment found across repeated randomized constructions followed by pairwise-swap local improvement.

The method is a heuristic: it can find useful layouts without enumerating every possible assignment, but it does not prove that the best layout found is globally optimal.

### Model assumptions and scope

- Each functional area is assigned to exactly one candidate location, and no location holds more than one area. Candidate locations may remain unused.
- An assignment is feasible only when the area's space requirement does not exceed the capacity of its assigned location.
- Locations are represented by points on a rectangular grid. Travel distance is rectilinear (Manhattan) distance; aisle geometry, congestion, and routes around obstacles are not modeled.
- Each flow is directed and measured in units per day. The objective weights the distance between two areas by the flow from the first area to the second.
- The construction phase processes areas in randomized order. For each area, it selects a feasible location uniformly at random with probability `alpha`; otherwise, it selects a location with the lowest incremental flow-weighted distance to areas already placed.
- The improvement phase considers exchanges between pairs of assigned areas. It accepts strict cost reductions that preserve capacity feasibility. It does not move an area to an unused location.

### Mathematical model

Let $A$ be the set of functional areas and $L$ the set of candidate locations. Let $f_{ab}$ be the directed flow from area $a$ to area $b$, $r_a$ the space required by area $a$, and $C_l$ the capacity of location $l$. Let $(x_l,y_l)$ be the coordinates of location $l$. The rectilinear distance between locations $l$ and $m$ is

$$d_{lm}=|x_l-x_m|+|y_l-y_m|.$$

The decision variable $z_{al}$ equals 1 when area $a$ is assigned to location $l$, and 0 otherwise. The model objective and assignment constraints are

$$\begin{aligned}
\min \quad & \sum_{a\in A}\sum_{b\in A}\sum_{l\in L}\sum_{m\in L} f_{ab}d_{lm}z_{al}z_{bm} \\
\text{subject to}\quad
& \sum_{l\in L}z_{al}=1 && \forall a\in A,\\
& \sum_{a\in A}z_{al}\leq 1 && \forall l\in L,\\
& z_{al}=0 && \text{if }r_a>C_l,\\
& z_{al}\in\{0,1\} && \forall a\in A,\ l\in L.
\end{aligned}$$

Here, $z_{al}z_{bm}$ contributes the flow-weighted distance when areas $a$ and $b$ occupy locations $l$ and $m$. The code represents the same assignment with a dictionary from area names to location indices rather than explicitly storing binary variables. `calculate_objective_value` evaluates the objective, while the construction and swap checks enforce the assignment and capacity constraints.

### How the heuristic works

During construction, let $S$ be the areas already assigned, and let $x_b$ be the location of an assigned area $b$. Placing the current area $a$ at candidate location $l$ adds the following cost for flows between $a$ and the assigned areas:

$$\Delta(a,l\mid S)=\sum_{b\in S}(f_{ab}+f_{ba})d_{l,x_b}.$$

The two flow terms are both needed because the objective includes directed flows in both directions. The greedy choice minimizes this incremental cost among feasible locations. With probability `alpha`, the construction instead chooses uniformly among feasible locations, adding diversification. Randomizing the area order and repeating construction provides additional variation.

After construction, pairwise-swap improvement examines pairs of assigned areas. A swap is considered only if each area fits in the other area's location. The swap is accepted when it strictly reduces the full objective. Repeated sweeps stop when no improving swap is found or the iteration cap is reached. A solution with no improving feasible pairwise swap is locally optimal with respect to this neighborhood; it is not necessarily globally optimal.

The method's runtime depends on the number of areas, candidate locations, and nonzero flows. A construction evaluates candidate locations using the areas already placed. In one improvement sweep, there are at most $O(|A|^2)$ candidate swaps, and each full objective evaluation takes $O(|E|)$ time, where $E$ is the set of flow arcs. Thus, the improvement phase takes $O(|A|^2|E|)$ per sweep, with at most 100 sweeps per construction. Repeated construction multiplies the work by the number of requested iterations. The heuristic avoids exhaustive enumeration of all feasible assignments, whose number can grow rapidly with the numbers of areas and locations.

### Interpreting the results

The reported objective is in flow-units multiplied by grid-distance units per day. A lower value is better for the stated model, but it is not directly a monetary cost unless flow and distance have been converted using an appropriate operational cost rate. The random-assignment baseline and convergence experiment are empirical comparisons for this run, not optimality certificates. Raw objective values from differently sized scalability instances are not directly comparable because their flows and numbers of areas differ.

In [ ]:
# Import the numerical, tabular, plotting, and reproducibility tools used below.
import math
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Fix both random-number generators so repeated runs follow the same random sequence.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class WarehouseLayoutHeuristic:
    """Randomized construction followed by pairwise-swap improvement."""

    def __init__(
        self,
        functional_areas,
        locations,
        flows,
        space_requirements,
        capacities,
        alpha=0.25,
    ):
        """Store a warehouse instance and precompute its distance matrix.

        `alpha` is the probability of choosing a feasible location uniformly
        at random during construction; otherwise, the incremental-cost
        criterion is used.
        """
        self.functional_areas = list(functional_areas)
        self.locations = list(locations)
        self.flows = dict(flows)
        self.space_requirements = dict(space_requirements)
        self.capacities = list(capacities)
        self.alpha = alpha

        if not 0 <= self.alpha <= 1:
            raise ValueError("alpha must be between 0 and 1.")
        if len(set(self.functional_areas)) != len(self.functional_areas):
            raise ValueError("Functional area names must be unique.")
        if len(self.locations) != len(self.capacities):
            raise ValueError("Each candidate location must have one capacity.")
        if len(self.functional_areas) > len(self.locations):
            raise ValueError("There must be at least as many locations as areas.")
        if any(area not in self.space_requirements for area in self.functional_areas):
            raise ValueError("A space requirement is missing for a functional area.")
        if any(
            area_from not in self.functional_areas
            or area_to not in self.functional_areas
            for area_from, area_to in self.flows
        ):
            raise ValueError("Every flow must refer to known functional areas.")

        self.distance_matrix = self._calculate_distances()

    def _calculate_distances(self):
        """Return the pairwise Manhattan-distance matrix for locations."""
        n_locations = len(self.locations)
        distances = np.zeros((n_locations, n_locations), dtype=float)

        for i, (x1, y1) in enumerate(self.locations):
            for j, (x2, y2) in enumerate(self.locations):
                distances[i, j] = abs(x1 - x2) + abs(y1 - y2)

        return distances

    def _calculate_location_preferences(self, area, assigned_locations):
        """Score candidates by the incremental cost to assigned areas.

        Higher scores are preferred, so the negative incremental cost is
        returned. The assigned locations are passed explicitly because the
        score depends on the partial layout built so far.
        """
        preferences = np.zeros(len(self.locations), dtype=float)

        for location_idx in range(len(self.locations)):
            incremental_cost = 0.0
            for other_area, other_location_idx in assigned_locations.items():
                flow_out = self.flows.get((area, other_area), 0)
                flow_in = self.flows.get((other_area, area), 0)
                distance = self.distance_matrix[location_idx, other_location_idx]
                incremental_cost += (flow_out + flow_in) * distance

            preferences[location_idx] = -incremental_cost

        return preferences

    def randomized_construction(self):
        """Build a feasible assignment using randomization and greedy scores."""
        assigned_locations = {}
        available_areas = self.functional_areas.copy()
        random.shuffle(available_areas)

        for area in available_areas:
            feasible_locations = [
                location_idx
                for location_idx in range(len(self.locations))
                if location_idx not in assigned_locations.values()
                and self.space_requirements[area] <= self.capacities[location_idx]
            ]

            if not feasible_locations:
                continue

            if random.random() < self.alpha:
                selected_location = random.choice(feasible_locations)
            else:
                location_scores = self._calculate_location_preferences(
                    area,
                    assigned_locations,
                )
                selected_location = max(
                    feasible_locations,
                    key=lambda location_idx: location_scores[location_idx],
                )

            assigned_locations[area] = selected_location

        return assigned_locations

    def calculate_objective_value(self, assignment):
        """Calculate total directed-flow-weighted Manhattan distance."""
        total_cost = 0.0

        for (area_from, area_to), flow_intensity in self.flows.items():
            if area_from in assignment and area_to in assignment:
                location_from = assignment[area_from]
                location_to = assignment[area_to]
                distance = self.distance_matrix[location_from, location_to]
                total_cost += flow_intensity * distance

        return total_cost

    def two_opt_improvement(self, assignment):
        """Improve an assignment by accepting feasible, cost-reducing swaps."""
        current_assignment = assignment.copy()
        current_cost = self.calculate_objective_value(current_assignment)
        improved = True
        iteration = 0

        while improved and iteration < 100:
            improved = False
            iteration += 1
            areas_list = list(current_assignment)

            for i in range(len(areas_list)):
                for j in range(i + 1, len(areas_list)):
                    area1 = areas_list[i]
                    area2 = areas_list[j]
                    location1 = current_assignment[area1]
                    location2 = current_assignment[area2]

                    feasible_swap = (
                        self.space_requirements[area1] <= self.capacities[location2]
                        and self.space_requirements[area2] <= self.capacities[location1]
                    )
                    if not feasible_swap:
                        continue

                    test_assignment = current_assignment.copy()
                    test_assignment[area1] = location2
                    test_assignment[area2] = location1
                    new_cost = self.calculate_objective_value(test_assignment)

                    is_strict_improvement = (
                        new_cost < current_cost
                        and not math.isclose(
                            new_cost,
                            current_cost,
                            rel_tol=1e-12,
                            abs_tol=1e-9,
                        )
                    )
                    if is_strict_improvement:
                        current_assignment = test_assignment
                        current_cost = new_cost
                        improved = True

        return current_assignment, current_cost

    def solve(self, max_iterations=1000):
        """Return the best complete feasible solution found across starts."""
        if max_iterations <= 0:
            raise ValueError("max_iterations must be positive.")

        best_assignment = None
        best_cost = float("inf")

        for _ in range(max_iterations):
            assignment = self.randomized_construction()
            if len(assignment) != len(self.functional_areas):
                continue

            improved_assignment, improved_cost = self.two_opt_improvement(assignment)
            is_better = improved_cost < best_cost and not math.isclose(
                improved_cost,
                best_cost,
                rel_tol=1e-12,
                abs_tol=1e-9,
            )
            if is_better:
                best_assignment = improved_assignment
                best_cost = improved_cost

        if best_assignment is None:
            raise ValueError(
                "No complete feasible assignment was found. Check capacities "
                "or increase the number of construction iterations."
            )

        return best_assignment, best_cost

In [ ]:
# Define the MegaCorp warehouse instance used in this example.
functional_areas = [
    "Receiving",
    "Fast_Storage",
    "Medium_Storage",
    "Slow_Storage",
    "Packing",
    "Shipping",
]

# The 12 candidate locations form a 4-by-3 grid.
locations = [
    (1, 3), (2, 3), (3, 3), (4, 3),
    (1, 2), (2, 2), (3, 2), (4, 2),
    (1, 1), (2, 1), (3, 1), (4, 1),
]

# Directed material flows are measured in units per day.
flows = {
    ("Receiving", "Fast_Storage"): 450,
    ("Receiving", "Medium_Storage"): 250,
    ("Receiving", "Slow_Storage"): 100,
    ("Fast_Storage", "Packing"): 400,
    ("Medium_Storage", "Packing"): 200,
    ("Slow_Storage", "Packing"): 80,
    ("Packing", "Shipping"): 680,
}

# Space requirements and capacities are measured in square feet.
space_requirements = {
    "Receiving": 800,
    "Fast_Storage": 1200,
    "Medium_Storage": 1000,
    "Slow_Storage": 1500,
    "Packing": 600,
    "Shipping": 700,
}
capacities = [2000] * 12

print("MegaCorp Warehouse Layout Optimization Setup:")
print(f"Functional areas: {len(functional_areas)}")
print(f"Candidate locations: {len(locations)} in a 4-by-3 grid")
print(f"Total daily flow: {sum(flows.values())} units/day")
print("Alpha parameter: 0.25 (25% probability of random feasible selection)")

### Solve the example

The solver runs the randomized construction and pairwise-swap improvement repeatedly. The returned assignment is the lowest-cost complete feasible solution found in those runs. The word “best” refers to the best solution found by this heuristic, not a proof of global optimality.

In [ ]:
heuristic = WarehouseLayoutHeuristic(
    functional_areas=functional_areas,
    locations=locations,
    flows=flows,
    space_requirements=space_requirements,
    capacities=capacities,
    alpha=0.25,
)

print("Running randomized construction with pairwise-swap improvement...")
start_time = time.time()
best_assignment, best_cost = heuristic.solve(max_iterations=1000)
computation_time = time.time() - start_time

assert len(best_assignment) == len(functional_areas)
assert len(set(best_assignment.values())) == len(best_assignment)
assert all(
    space_requirements[area] <= capacities[location_idx]
    for area, location_idx in best_assignment.items()
)
assert math.isclose(
    best_cost,
    heuristic.calculate_objective_value(best_assignment),
    rel_tol=1e-12,
    abs_tol=1e-9,
)

print("\n=== HEURISTIC SOLUTION RESULTS ===")
print(f"Computation time: {computation_time:.3f} seconds")
print(f"Best solution found: {best_cost:.1f} flow-distance units/day")
print("Assignment:")

for area, location_idx in best_assignment.items():
    print(f"  {area}: Location {location_idx} at {locations[location_idx]}")

### Compare with random feasible assignments

A random-assignment baseline helps show how the heuristic performs relative to unoptimized feasible layouts. The baseline uses the same location-capacity and one-area-per-location rules. Its sampled average and best value are descriptive statistics from a finite sample, not guarantees about all random assignments.

In [ ]:
def random_assignment_baseline(heuristic_instance):
    """Construct one feasible assignment using random choices only."""
    areas = heuristic_instance.functional_areas.copy()
    random.shuffle(areas)
    available_locations = set(range(len(heuristic_instance.locations)))
    assignment = {}

    for area in areas:
        feasible_locations = [
            location_idx
            for location_idx in available_locations
            if heuristic_instance.space_requirements[area]
            <= heuristic_instance.capacities[location_idx]
        ]
        if not feasible_locations:
            return None

        selected_location = random.choice(feasible_locations)
        assignment[area] = selected_location
        available_locations.remove(selected_location)

    return assignment


random_costs = []
for _ in range(100):
    random_assignment = random_assignment_baseline(heuristic)
    if random_assignment is not None:
        random_costs.append(
            heuristic.calculate_objective_value(random_assignment)
        )

if not random_costs:
    raise ValueError("The baseline did not produce any feasible assignments.")

avg_random_cost = float(np.mean(random_costs))
best_random_cost = min(random_costs)
worst_random_cost = max(random_costs)
improvement_over_avg = (
    (avg_random_cost - best_cost) / avg_random_cost * 100
    if avg_random_cost > 0
    else 0.0
)
improvement_over_best_random = (
    (best_random_cost - best_cost) / best_random_cost * 100
    if best_random_cost > 0
    else 0.0
)

print("=== SOLUTION QUALITY ANALYSIS ===")
print("Random feasible assignment baseline:")
print(f"  Sample average: {avg_random_cost:.1f} flow-distance units/day")
print(f"  Best sampled: {best_random_cost:.1f} flow-distance units/day")
print(f"  Worst sampled: {worst_random_cost:.1f} flow-distance units/day")
print("\nHeuristic solution:")
print(f"  Best found: {best_cost:.1f} flow-distance units/day")
print("\nDifference relative to the sampled baselines:")
print(f"  Versus sample average: {improvement_over_avg:.1f}% lower")
print(f"  Versus best sampled assignment: {improvement_over_best_random:.1f}% lower")

### Visualize the assignment and flows

The floor-plan panel places each area at its assigned grid coordinate. Red arrows indicate directed material flows, with line width proportional to flow volume. The heatmap shows the same directed flow data: rows are origin areas and columns are destination areas. The objective value displayed in the title is the flow-weighted distance for the assignment.

In [ ]:
def visualize_heuristic_layout(
    assignment,
    cost,
    locations,
    functional_areas,
    flows,
    title="Heuristic Warehouse Layout",
):
    """Plot the assigned floor plan and the directed flow matrix."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

    colors = {
        "Receiving": "#FF6B6B",
        "Fast_Storage": "#4ECDC4",
        "Medium_Storage": "#45B7D1",
        "Slow_Storage": "#96CEB4",
        "Packing": "#FECA57",
        "Shipping": "#FF9FF3",
    }

    ax1.set_title(
        f"{title}\n(Objective: {cost:.1f} flow-distance units/day)",
        fontsize=14,
        fontweight="bold",
    )

    for x in range(5):
        ax1.axvline(x, color="gray", linestyle="--", alpha=0.3)
    for y in range(4):
        ax1.axhline(y, color="gray", linestyle="--", alpha=0.3)

    for area, location_idx in assignment.items():
        x, y = locations[location_idx]
        rectangle = plt.Rectangle(
            (x - 0.9, y - 0.9),
            1.8,
            1.8,
            facecolor=colors[area],
            edgecolor="black",
            linewidth=2,
            alpha=0.7,
        )
        ax1.add_patch(rectangle)
        ax1.text(
            x,
            y,
            area.replace("_", " "),
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            wrap=True,
        )

    for (area_from, area_to), flow_intensity in flows.items():
        if flow_intensity <= 0 or area_from not in assignment or area_to not in assignment:
            continue

        x_from, y_from = locations[assignment[area_from]]
        x_to, y_to = locations[assignment[area_to]]
        ax1.annotate(
            "",
            xy=(x_to, y_to),
            xytext=(x_from, y_from),
            arrowprops={
                "arrowstyle": "->",
                "lw": flow_intensity / 150,
                "color": "red",
                "alpha": 0.6,
            },
        )

    ax1.set_xlim(0, 5)
    ax1.set_ylim(0, 4)
    ax1.set_aspect("equal")
    ax1.set_xlabel("Grid x-coordinate")
    ax1.set_ylabel("Grid y-coordinate")
    ax1.text(
        0.02,
        0.02,
        "Red arrows show flow direction; width is proportional to units/day.",
        transform=ax1.transAxes,
        fontsize=8,
        va="bottom",
    )

    ax2.set_title("Directed Material Flow (units/day)", fontsize=14, fontweight="bold")
    flow_matrix = np.zeros((len(functional_areas), len(functional_areas)))
    area_labels = [area.replace("_", "\n") for area in functional_areas]

    for (area_from, area_to), flow_intensity in flows.items():
        i = functional_areas.index(area_from)
        j = functional_areas.index(area_to)
        flow_matrix[i, j] = flow_intensity

    image = ax2.imshow(flow_matrix, cmap="Reds", aspect="auto")
    ax2.set_xticks(range(len(functional_areas)))
    ax2.set_yticks(range(len(functional_areas)))
    ax2.set_xticklabels(area_labels, rotation=45, ha="right")
    ax2.set_yticklabels(area_labels)
    ax2.set_xlabel("Destination area")
    ax2.set_ylabel("Origin area")

    for i in range(len(functional_areas)):
        for j in range(len(functional_areas)):
            if flow_matrix[i, j] > 0:
                ax2.text(
                    j,
                    i,
                    f"{flow_matrix[i, j]:.0f}",
                    ha="center",
                    va="center",
                    fontweight="bold",
                    color="darkred",
                )

    colorbar = plt.colorbar(image, ax=ax2)
    colorbar.set_label("Flow (units/day)")

    legend_elements = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            facecolor=colors[area],
            alpha=0.7,
            label=area.replace("_", " "),
        )
        for area in functional_areas
    ]
    ax1.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(1.02, 1))

    plt.tight_layout()
    plt.show()


visualize_heuristic_layout(
    best_assignment,
    best_cost,
    locations,
    functional_areas,
    flows,
    title="Best Layout Found by the Heuristic",
)

### Convergence experiment

This experiment records the best cost found so far over 200 independent construction-and-improvement runs. A nonincreasing best-so-far curve is expected by definition: each point retains the best result from all runs up to that point. The comparison with the random baseline average indicates when a run's best-so-far value falls below a chosen reference threshold; it is not a proof of convergence to a global optimum.

In [ ]:
def convergence_analysis(heuristic_instance, n_iterations=200):
    """Track the best cost found so far across repeated heuristic runs."""
    costs_over_time = []
    best_so_far = float("inf")

    for _ in range(n_iterations):
        assignment = heuristic_instance.randomized_construction()
        if len(assignment) == len(heuristic_instance.functional_areas):
            _, improved_cost = heuristic_instance.two_opt_improvement(assignment)
            best_so_far = min(best_so_far, improved_cost)

        costs_over_time.append(best_so_far)

    return costs_over_time


convergence_costs = convergence_analysis(heuristic)
if not math.isfinite(convergence_costs[-1]):
    raise ValueError("The convergence experiment found no complete feasible assignment.")

plt.figure(figsize=(12, 6))
plt.plot(convergence_costs, color="blue", linewidth=2, label="Best cost so far")
plt.axhline(
    y=best_cost,
    color="red",
    linestyle="--",
    label=f"Best cost in main run: {best_cost:.1f}",
)
plt.axhline(
    y=avg_random_cost,
    color="green",
    linestyle=":",
    label=f"Random-baseline sample average: {avg_random_cost:.1f}",
)
plt.xlabel("Construction-and-improvement run")
plt.ylabel("Best objective so far (flow-distance units/day)")
plt.title("Heuristic Best-So-Far Progress")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

threshold_90_percent = 0.9 * avg_random_cost
iterations_to_threshold = next(
    (
        i
        for i, cost in enumerate(convergence_costs)
        if cost <= threshold_90_percent
    ),
    None,
)
within_one_percent_of_main_best = best_cost * 1.01
iterations_within_one_percent = next(
    (
        i
        for i, cost in enumerate(convergence_costs)
        if cost <= within_one_percent_of_main_best
    ),
    None,
)

print("Convergence experiment metrics:")
if iterations_to_threshold is None:
    print("  The 90%-of-baseline-average threshold was not reached.")
else:
    print(
        "  Runs until the best-so-far cost reached 90% of the random-baseline "
        f"average: {iterations_to_threshold}"
    )
if iterations_within_one_percent is None:
    print("  The main-run best-cost reference was not reached within 1%.")
else:
    print(
        "  Runs until the best-so-far cost was within 1% of the main-run "
        f"best cost: {iterations_within_one_percent}"
    )
print(f"  Final best-so-far cost: {convergence_costs[-1]:.1f}")

### Scalability experiment

The following experiment applies the same heuristic to generated instances with increasing numbers of areas and locations. These instances use randomly generated flows and space requirements, so their objective values are on different scales and should not be read as a solution-quality ranking. The timing measurements are specific to the machine and software environment running the notebook.

In [ ]:
def scalability_test():
    """Measure heuristic runtime and cost on generated instance sizes."""
    test_results = []
    test_sizes = [(4, 8), (6, 12), (8, 16), (10, 20)]

    for n_areas, n_locations in test_sizes:
        test_areas = [f"Area_{i}" for i in range(n_areas)]
        test_locations = [
            (i % 4 + 1, i // 4 + 1)
            for i in range(n_locations)
        ]
        test_flows = {}

        for i in range(n_areas):
            for j in range(n_areas):
                if i != j and random.random() < 0.3:
                    test_flows[(test_areas[i], test_areas[j])] = random.randint(50, 500)

        test_space_requirements = {
            area: random.randint(500, 1500)
            for area in test_areas
        }
        test_capacities = [2000] * n_locations

        test_heuristic = WarehouseLayoutHeuristic(
            functional_areas=test_areas,
            locations=test_locations,
            flows=test_flows,
            space_requirements=test_space_requirements,
            capacities=test_capacities,
            alpha=0.25,
        )

        start_time = time.time()
        test_assignment, test_cost = test_heuristic.solve(max_iterations=500)
        elapsed_time = time.time() - start_time

        test_results.append(
            {
                "areas": n_areas,
                "locations": n_locations,
                "cost": test_cost,
                "time": elapsed_time,
            }
        )

    return test_results


scalability_results = scalability_test()
area_counts = [result["areas"] for result in scalability_results]
runtimes = [result["time"] for result in scalability_results]
objective_values = [result["cost"] for result in scalability_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.plot(area_counts, runtimes, "bo-", linewidth=2, markersize=8)
ax1.set_xlabel("Number of functional areas")
ax1.set_ylabel("Computation time (seconds)")
ax1.set_title("Runtime by Instance Size")
ax1.grid(True, alpha=0.3)
ax1.set_xticks(area_counts)

ax2.plot(area_counts, objective_values, "ro-", linewidth=2, markersize=8)
ax2.set_xlabel("Number of functional areas")
ax2.set_ylabel("Objective (flow-distance units/day)")
ax2.set_title("Raw Objective Values by Instance Size")
ax2.grid(True, alpha=0.3)
ax2.set_xticks(area_counts)

plt.tight_layout()
plt.show()

print("Scalability experiment results:")
for result in scalability_results:
    print(
        f"  {result['areas']} areas, {result['locations']} locations: "
        f"{result['time']:.3f} seconds, "
        f"objective {result['cost']:.1f} flow-distance units/day"
    )

## Strengths, limitations, and use

### Strengths

- Randomized construction can generate diverse feasible starting assignments without enumerating all layouts.
- The greedy placement score uses the directed flow-weighted distance contributed by areas already assigned.
- Pairwise exchanges provide a transparent local-improvement step, and capacity checks preserve feasibility.
- Repeated starts allow the method to explore multiple constructed layouts while retaining the best result found.

### Limitations

- The method provides no global optimality guarantee. Its result depends on construction order, randomized choices, `alpha`, and the swap neighborhood.
- Pairwise swaps only exchange the locations of two assigned areas. They do not move an area to an unused location, and a local optimum may therefore have improving moves outside this neighborhood.
- Construction can produce an incomplete assignment when earlier choices leave no feasible location for a later area. Incomplete starts are skipped; the solver reports an error if no complete feasible start is found.
- Results depend on the modeled Manhattan geometry and flow data. Real facilities may require aisle networks, handling costs, area shapes, safety rules, or operational constraints not represented here.
- Runtime measurements and raw objective values from generated instances are specific to the experimental setup; objective values across different instances are not directly comparable.

This approach is useful for teaching randomized construction and local search, and for obtaining candidate layouts when a quick heuristic result is appropriate. Before operational use, the model assumptions and feasibility requirements should be checked against the facility being designed.